# Milestone 1: Retrieval & Evaluation

This notebook covers:
1. Building and testing BM25, semantic, and hybrid retrievers
2. Qualitative evaluation across a query set

In [1]:
import sys
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
INDICES_DIR = PROJECT_ROOT / "indices"

from src.utils import load_corpus
from src.bm25 import BM25Retriever
from src.semantic import SemanticRetriever
from src.hybrid import HybridRetriever

# Load corpus
corpus = load_corpus(str(DATA_PROCESSED / "product_corpus.parquet"))
print(f"Corpus loaded: {len(corpus):,} products")


c:\Users\amato\miniforge3\envs\575-project\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Corpus loaded: 112,578 products


## 1. BM25 Retriever

Building a keyword-based retrieval system using BM25 (Okapi BM25).

In [2]:
bm25 = BM25Retriever()
bm25.build_index(corpus)
bm25.save(str(INDICES_DIR / "bm25_index.pkl"))
print("BM25 index built and saved.")

results = bm25.search("vitamin C serum", top_k=5)
print("\nTest query: 'vitamin C serum'")
for i, r in enumerate(results):
    print(f"{i+1}. {r['title']} (score: {r['score']:.4f})")

BM25 index built and saved.

Test query: 'vitamin C serum'
1. Organic Vitamin C Serum for Face-Professional Strength-Organic Vitamin C Serum-Hyaluronic Acid-Vitamin E-Naturally Derived 20% Vitamin C The Best Vitamin C Serum (1 Ounce) (score: 21.4679)
2. 24K Vitamin C Serum for Face 2 PACK, Facial Serum with Vitamin C and Hyaluronic Acid, Facial Moisturizer 2 Pack Vitamin C Serum (score: 21.0783)
3. Renew Vitamin C Serum (score: 20.8527)
4. Vitamin C Plus Serum (score: 20.8527)
5. Vitamin C Serum 1oz (score: 20.8527)


**Observations on BM25 results for "vitamin C serum":**

- BM25 performs well on this query because the search terms appear verbatim in product titles (hence "keyword retrieval").
- The top two results score higher (~21.1–21.5) because they repeat "vitamin C serum" multiple times in the title, which BM25 rewards via term frequency.
- Results 3–5 all tie at ~20.85, because each contains the exact phrase once in a short title with little other text to dilute the score.
- A limitation is visible here which is that BM25 has no understanding of *meaning*. A query like "face brightening serum with antioxidants" would miss these same products even though vitamin C serums are exactly that. This is the gap we expect semantic retrieval to fill.

## 2. Semantic Retriever

Building a semantic search system using sentence-transformers (all-MiniLM-L6-v2) and FAISS.

In [3]:
semantic = SemanticRetriever()
semantic.build_index(corpus)
semantic.save(str(INDICES_DIR / "faiss_index"))
print("Semantic index built and saved.")

results = semantic.search("vitamin C serum", top_k=5)
print("\nTest query: 'vitamin C serum'")
for i, r in enumerate(results):
    print(f"  {i+1}. {r['title']} (score: {r['score']:.4f})")

print("\n---")
results2 = semantic.search("something to protect from sun damage", top_k=5)
print("\nTest query: 'something to protect from sun damage'")
for i, r in enumerate(results2):
    print(f"  {i+1}. {r['title']} (score: {r['score']:.4f})")

print("\n---")
results3 = semantic.search("face brightening serum with antioxidants", top_k=5)
print("\nTest query: 'face brightening serum with antioxidants'")
for i, r in enumerate(results3):
    print(f"  {i+1}. {r['title']} (score: {r['score']:.4f})")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1907.51it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Semantic index built and saved.

Test query: 'vitamin C serum'
  1. PURE VITAMIN C SERUM (score: 0.9416)
  2. Vitamin C Plus Serum (score: 0.9233)
  3. Vitamin C Serum 1oz (score: 0.8743)
  4. Renew Vitamin C Serum (score: 0.8461)
  5. RA Herbals Ultimate Vitamin C Serum (score: 0.7833)

---

Test query: 'something to protect from sun damage'
  1. Near Skin Dustless Defense Sun Block (score: 0.6586)
  2. UV Capture PLUS Pure Mild Sun Cream Sun Block Sunscreen SPF PA +++ + Moistful Essential Sun Block Cream With Skin Relieving Protection Barrier (score: 0.5880)
  3. [The Saem] Eco Earth Power Sun Cream 50g (Waterproof Sun Block) (score: 0.5658)
  4. Banana Boat Sunscreen Simply Protect Kids Tear Free, Broad Spectrum Mineral Sunscreen Spray, TSA Approved Size, SPF 50+, 1.8 oz (score: 0.5587)
  5. Solarbiome Ampule Sunscreen Sun Block Sunscreen SPF + 50 PA +++ Dust Blocking UV Protector, 50ml (score: 0.5275)

---

Test query: 'face brightening serum with antioxidants'
  1. Majestic Pure S

**Observations on semantic retriever results:**

- `"vitamin C serum"`: Semantic search returns relevant results (all are vitamin C serums) but ranks them differently than BM25. "PURE VITAMIN C SERUM" scores highest (0.94) despite having a short title because the embedding captures meaning rather than rewarding keyword repetition. BM25 favoured longer titles that repeated the phrase multiple times.
- `"something to protect from sun damage"`: None of these results contain the exact words "protect from sun damage" yet the model correctly retrieves sunscreen/sun block products by understanding the *intent* behind the query. BM25 would struggle here because the query terms don't match product vocabulary (e.g., "SPF", "sun block", "sunscreen").
- `"face brightening serum with antioxidants"`: Semantic search retrieves a mix of brightening serums and antioxidant products, including a Vitamin C serum (result 5), which is contextually correct since vitamin C is an antioxidant. This demonstrates the model's ability to connect related concepts that don't share exact keywords.
- Scores range from ~0.5 to ~0.94 (cosine similarity). Exact keyword matches score highest. Intent-based queries score lower but still retrieve relevant products. This is expected because the model has to "bridge" a larger semantic gap for paraphrased queries.
- `Limitation:`: The model returned a duplicate (Trader Joe's Antioxidant Serum appears twice) which suggests duplicate or near-duplicate products exist in the corpus. Deduplication could improve result diversity.

## 3. Hybrid Retriever

Combining BM25 and semantic scores with weighted linear combination. The hybrid approach min-max normalizes both score sets to [0, 1], then combines them: `score = bm25_weight * bm25_score + (1 - bm25_weight) * semantic_score`.

In [4]:
hybrid = HybridRetriever(bm25, semantic)

# Test with same queries used in BM25 and semantic sections
print("Test query: 'vitamin C serum' (weight=0.5)")
results = hybrid.search("vitamin C serum", top_k=5, bm25_weight=0.5)
for i, r in enumerate(results):
    print(f"  {i+1}. {r['title']} (score: {r['score']:.4f})")

print("\n---")
print("\nTest query: 'something to protect from sun damage' (weight=0.5)")
results2 = hybrid.search("something to protect from sun damage", top_k=5, bm25_weight=0.5)
for i, r in enumerate(results2):
    print(f"  {i+1}. {r['title']} (score: {r['score']:.4f})")

print("\n---")
print("\nTest query: 'gentle cleanser for sensitive skin' (weight=0.3, favor semantic)")
results3 = hybrid.search("gentle cleanser for sensitive skin", top_k=5, bm25_weight=0.3)
for i, r in enumerate(results3):
    print(f"  {i+1}. {r['title']} (score: {r['score']:.4f})")

Test query: 'vitamin C serum' (weight=0.5)
  1. PURE VITAMIN C SERUM (score: 0.8697)
  2. Vitamin C Plus Serum (score: 0.8295)
  3. Vitamin C Serum 1oz (score: 0.7219)
  4. Renew Vitamin C Serum (score: 0.6600)
  5. Organic Vitamin C Serum for Face-Professional Strength-Organic Vitamin C Serum-Hyaluronic Acid-Vitamin E-Naturally Derived 20% Vitamin C The Best Vitamin C Serum (1 Ounce) (score: 0.5000)

---

Test query: 'something to protect from sun damage' (weight=0.5)
  1. CELLFUSION C: Derma Relief Sun Screen 100 SPF 50+/PA ++++, 1+1 50ML+50ML: Sooth sensitive Skin, Effective protect from UV rays. (score: 0.5000)
  2. Near Skin Dustless Defense Sun Block (score: 0.5000)
  3. Bare Minerals Something to Talk A-Pout Mini Moxie Plumping Lipgloss Set (score: 0.4651)
  4. A-derma Protect Ah Repairing Milk After Sun 250ml (score: 0.4019)
  5. KOSE Sekkisei Sun Protect Essence Milk SPF50+ / PA++++ 1.9oz, 57ml (score: 0.2770)

---

Test query: 'gentle cleanser for sensitive skin' (weight=0.3,

**Observations on hybrid retriever results:**

- `"vitamin C serum" (weight=0.5)`: The hybrid reranks results compared to both individual methods. "PURE VITAMIN C SERUM" stays at #1 (score 0.87) because it scores well on both keyword match and semantic similarity. Note that the long-titled product that BM25 ranked #1 (Organic Vitamin C Serum...) drops to #5 (score 0.50). It had the highest BM25 score from keyword repetition but its semantic score was lower and normalization penalizes the gap. The hybrid balances precision (exact match) with meaning.
- `"something to protect from sun damage" (weight=0.5)`: The top product (CELLFUSION C sun screen) wasn't #1 in either individual method but it scored reasonably well in *both*, so the combined score pushed it to the top. This shows hybrid's strength as it surfaces products that are good across both dimensions rather than extreme in one. However, "Bare Minerals Something to Talk A-Pout" at #3 is a false positive. BM25 matched the word "something" literally, inflating its hybrid score despite being irrelevant.
- `"gentle cleanser for sensitive skin" (weight=0.3, favoring semantic)`: Lowering the BM25 weight to 0.3 lets semantic similarity dominate. The top result (EveryShine Rose Mousse Foam Cleanser) scores 0.74, well above the rest, which suggests strong alignment in both methods. The results are all genuinely relevant cleansers for sensitive skin.
- Score range is [0, 1] as expected from min-max normalization. The weight parameter gives a "tunable knob" which is useful in the app where users can adjust based on whether their query is keyword-heavy or intent-heavy.

## 4. Qualitative Evaluation

Running all three retrieval methods on our 21-query evaluation set and comparing results side-by-side.

In [5]:
queries_df = pd.read_csv(DATA_PROCESSED / "ground_truth.csv")

eval_results = []
for _, row in queries_df.iterrows():
    query = row["query"]
    bm25_res = bm25.search(query, top_k=5)
    sem_res = semantic.search(query, top_k=5)
    hyb_res = hybrid.search(query, top_k=5, bm25_weight=0.5)

    eval_results.append({
        "query_id": row["query_id"],
        "query": query,
        "difficulty": row["difficulty"],
        "bm25_top3": [r["title"][:50] for r in bm25_res[:3]],
        "semantic_top3": [r["title"][:50] for r in sem_res[:3]],
        "hybrid_top3": [r["title"][:50] for r in hyb_res[:3]],
    })

eval_df = pd.DataFrame(eval_results)
print(f"Evaluated {len(eval_df)} queries across 3 methods")
eval_df

Evaluated 21 queries across 3 methods


,query_id,query,difficulty,bm25_top3,semantic_top3,hybrid_top3
0,1,vitamin C serum,easy,[Organic Vitamin C Serum for Face-Professional...,"[PURE VITAMIN C SERUM, Vitamin C Plus Serum, V...","[PURE VITAMIN C SERUM, Vitamin C Plus Serum, V..."
1,2,coconut oil shampoo,easy,"[Oliology Coconut Oil Dry Shampoo, 8 Oz., FREY...","[Oliology Coconut Oil Dry Shampoo, 8 Oz., Sham...","[Oliology Coconut Oil Dry Shampoo, 8 Oz., Shea..."
2,3,charcoal face mask,easy,"[Charcoal Gel Face Mask, Aliceva Black Mask, B...","[Charcoal Gel Face Mask, Black Mask (Charcoal ...","[Charcoal Gel Face Mask, Beauty Foundry Charco..."
3,4,retinol cream,easy,"[Mererke_Pretty Retinol Cream, Moisturizing Cr...",[Pure Biology Night Cream with Retinol for Men...,[Retinol Plus Anti Aging Day cream with Retino...
4,5,sunscreen SPF 50,easy,[Cerave Sunscreen Bundle SPF 50 | Contains Min...,[Perfectly Plain Collection Sunscreen with Spf...,"[Anti-aging sunscreen SPF 50+ Duolys, 50 ml, A..."
5,6,tea tree face wash,easy,"[Tea Tree Face Wash 16 fl.oz. Seed + Posy, Cla...","[TISSERAND Tea Tree & Aloe Foaming Face Wash, ...","[Tea Tree Face Wash 16 fl.oz. Seed + Posy, Cla..."
6,7,argan oil hair mask,easy,[Spa Life Argan Black Mask - Blackhead Remover...,[Hair Repair Mask – Infused with Argan Oil for...,[Spa Life Argan Black Mask - Blackhead Remover...
7,8,something to reduce acne scars,medium,"[Crocodile Repair Face Serum for Acne Scars, A...",[Natural Acne Scar Removal Cream for Acne Scar...,[Natural Acne Scar Removal Cream for Acne Scar...
8,9,gift set that smells like vanilla,medium,"[Duke Cannon""Big Ass Tank Soap"" Smells Like Vi...","[Wild Spirit Chill Holiday Perfume Gift Set, D...","[Duke Cannon""Big Ass Tank Soap"" Smells Like Vi..."
9,10,product to make hair less frizzy,medium,[O BOTICARIO Match Frizz Patrol Shielding Hair...,[Herbal Essences Shampoo None of Your Frizznes...,[O BOTICARIO Match Frizz Patrol Shielding Hair...


In [6]:
# Side-by-side comparison for 5 selected queries (mix of easy, medium, hard)
sample_queries = [
    "vitamin C serum",                                  # easy
    "sunscreen SPF 50",                                 # easy
    "something to reduce acne scars",                   # medium
    "gentle cleanser for sensitive skin",               # medium
    "best anti-aging routine for dry skin under $25",   # hard
]

for query in sample_queries:
    difficulty = queries_df[queries_df["query"] == query]["difficulty"].values[0]
    print(f"\n{'='*90}")
    print(f"Query: '{query}' [{difficulty}]")
    print(f"{'='*90}")

    bm25_res = bm25.search(query, top_k=5)
    sem_res = semantic.search(query, top_k=5)

    print(f"\n  {'#':<3} {'BM25 Results':<45} {'Semantic Results'}")
    print(f"  {'---':<3} {'-'*44} {'-'*44}")
    for i in range(5):
        b_title = bm25_res[i]["title"][:42] if i < len(bm25_res) else "N/A"
        b_score = f"({bm25_res[i]['score']:.2f})" if i < len(bm25_res) else ""
        s_title = sem_res[i]["title"][:42] if i < len(sem_res) else "N/A"
        s_score = f"({sem_res[i]['score']:.2f})" if i < len(sem_res) else ""
        print(f"  {i+1:<3} {b_title:<35} {b_score:<8} {s_title:<35} {s_score}")


Query: 'vitamin C serum' [easy]

  #   BM25 Results                                  Semantic Results
  --- -------------------------------------------- --------------------------------------------
  1   Organic Vitamin C Serum for Face-Professio (21.47)  PURE VITAMIN C SERUM                (0.94)
  2   24K Vitamin C Serum for Face 2 PACK, Facia (21.08)  Vitamin C Plus Serum                (0.92)
  3   Renew Vitamin C Serum               (20.85)  Vitamin C Serum 1oz                 (0.87)
  4   Vitamin C Plus Serum                (20.85)  Renew Vitamin C Serum               (0.85)
  5   Vitamin C Serum 1oz                 (20.85)  RA Herbals Ultimate Vitamin C Serum (0.78)

Query: 'sunscreen SPF 50' [easy]

  #   BM25 Results                                  Semantic Results
  --- -------------------------------------------- --------------------------------------------
  1   Cerave Sunscreen Bundle SPF 50 | Contains  (24.09)  Perfectly Plain Collection Sunscreen with  (0.85)
  2   Ant

### Discussion & Comparison: BM25 vs Semantic vs Hybrid

**Per-Query Analysis (5 selected queries):**

1. **"vitamin C serum" [easy]** — Both methods retrieve relevant products. BM25 ranks the longest title first (more keyword repetitions = higher term frequency), while semantic ranks "PURE VITAMIN C SERUM" #1 for direct meaning alignment. Hybrid combines both signals effectively. **Winner: Tie** — easy keyword queries work well for both.

2. **"sunscreen SPF 50" [easy]** — BM25 is more precise: all top-5 contain exactly "SPF 50." Semantic retrieves SPF 40 and SPF 30 products because it understands "sunscreen" conceptually but cannot enforce numeric constraints. **Winner: BM25** — exact specification matching favors keyword search.

3. **"something to reduce acne scars" [medium]** — BM25 returns duplicates (Crocodile Repair Serum appears twice) and is distracted by "something." Semantic returns a more diverse set of scar-treatment products and correctly includes a general "Scar Removal Cream" without the word "acne." **Winner: Semantic** — handles vague phrasing and retrieves more diverse results.

4. **"gentle cleanser for sensitive skin" [medium]** — BM25 finds products explicitly labeled "gentle." Semantic surfaces products that *imply* gentleness ("Rose Mousse Foam," "Calming Chamomile") but also retrieves "Glycolic Cleanser" — glycolic acid is typically not gentle, showing the model doesn't fully understand ingredient implications. **Winner: BM25** — keyword precision matters when the query uses specific product vocabulary.

5. **"best anti-aging routine for dry skin under $25" [hard]** — Both struggle. BM25 matches on individual words: "under" triggers an eye cream for "dark circles *under* eyes," and "routine" matches "Men's Routine Pheromone Cream." Semantic finds more topically relevant anti-aging products but ignores the price constraint entirely. **Winner: Neither** — multi-constraint queries exceed both methods' capabilities.

---

**Where BM25 Fails but Semantic Succeeds:**
- **Intent-based queries** (e.g., "something to protect from sun damage"): BM25 cannot bridge the vocabulary gap between "protect" and "SPF/sunscreen." Semantic maps intent correctly.
- **Paraphrased queries** (e.g., "product to make hair less frizzy"): BM25 needs exact overlap; semantic connects "less frizzy" to anti-frizz products.
- **Conversational filler** (e.g., "something to keep skin hydrated all day"): The word "something" is noise for BM25 but invisible to the embedding model.

**Where Semantic Fails:**
- **Exact specifications**: Retrieves SPF 30 for an "SPF 50" query — embeddings treat numbers as semantically similar.
- **Ingredient-level reasoning**: Surfaces "Glycolic Cleanser" for a "gentle cleanser" query — the model doesn't understand that glycolic acid is harsh.
- **Structured constraints**: "Under $25" is understood conceptually but cannot be enforced without metadata filtering.

---

**Performance by Query Difficulty:**

| Difficulty | BM25 | Semantic | Hybrid |
|------------|------|----------|--------|
| **Easy** (exact keywords) | Strong — direct keyword match | Strong — but may include wrong specs | Strong — combines both signals |
| **Medium** (paraphrased/intent) | Mixed — fails on vague phrasing | Better — captures intent without keyword overlap | Best — semantic handles intent, BM25 boosts exact matches |
| **Hard** (multi-constraint) | Weak — false positives from individual word matches | Moderate — captures theme but ignores structured constraints | Moderate — inherits limitations of both |

**Key Takeaways:**
1. **BM25 excels at precision** for keyword-specific queries but fails when users paraphrase or describe needs conversationally.
2. **Semantic search excels at recall** for intent-based queries but cannot enforce exact specifications or structured constraints like price.
3. **Hybrid retrieval provides the best balance**, surfacing products that score well across both dimensions.
4. **Neither method handles multi-constraint queries well** (product type + skin type + budget). This is the gap a RAG pipeline could fill — an LLM can reason over retrieved candidates and filter on structured attributes like price and ratings.